# Workflows and Agents

---

## Core Distinction

| | Workflows | Agents |
|---|---|---|
| **Definition** | A series of calls to Claude through a **predetermined** series of steps | Claude is given a goal + tools and **figures out the steps itself** |
| **Control** | You define the flow in code | Claude decides what to do next |
| **Use when** | You can picture the exact steps needed | You're not sure what task or parameters Claude will face |
| **Example** | Image → CAD pipeline | Claude Code, Computer Use |

---

## Workflow Example — Image to CAD

**App:** User drags and drops an image of a metal part → system generates a STEP file

Because we know exactly what steps to take for any image input, this is a **workflow**:
```
1. Feed image to Claude → ask it to describe the object
         |
         v
2. Use the description → ask Claude to model it with CadQuery library
         |
         v
3. Create a rendering of the model
         |
         v
4. Ask Claude to grade rendering against original image
         |
   Issues found? → feed feedback back to step 2 (fix and retry)
         |
   Accepted? → Done
```

---

## Workflow Type — Evaluator-Optimizer Pattern
```
Input
  |
  v
Producer  ←──── Feedback ────┐
  |                           |
  | Submission                |
  v                           |
Grader ──── Not accepted ────┘
  |
  | Accepted
  v
Output
```

| Role | What it does |
|---|---|
| **Producer** | Creates the output (Claude models the part, generates rendering) |
| **Grader** | Evaluates the output against criteria (Claude compares rendering to original image) |
| **Feedback loop** | If rejected, grader sends feedback → producer improves and resubmits |

---

## Why Learn Workflow Patterns

> Workflow patterns are **repeatable recipes** — not just abstract concepts.  
> Once you know the Evaluator-Optimizer pattern, you can apply it to any
> feature that benefits from produce → grade → refine cycles.

Knowing the pattern name helps you recognise when to apply it and gives you
a proven structure to implement rather than starting from scratch.

---

## Key Takeaway

> **Workflows** = you write the steps in code, Claude executes each one.  
> **Agents** = you give Claude a goal and tools, Claude writes its own steps.  
> Choose workflows when the task is predictable. Choose agents when it isn't.

# Workflow Type: Parallelization

---

## The Problem with Single Complex Prompts

Asking Claude to evaluate a part against metal, polymer, ceramic, composite,
elastomer, and wood **all in one prompt** creates problems:

- Claude must juggle all criteria simultaneously
- Results are less focused and less reliable
- Hard to debug or improve a specific material's evaluation
- Adding a new material requires rewriting the whole prompt

---

## The Solution — Split and Run in Parallel

Send the same image to Claude **multiple times simultaneously**, each with
specialized criteria for one material only:
```
Image + Metal Criteria    → Claude → Metal Analysis Result
Image + Polymer Criteria  → Claude → Polymer Analysis Result   (all in parallel)
Image + Ceramic Criteria  → Claude → Ceramic Analysis Result
Image + Composite Criteria→ Claude → Composite Analysis Result
                                            |
                                            v
                                    Claude (Aggregator)
                                            |
                                            v
                               Final Material Recommendation
```

---

## Pattern Structure
```
User Task
    |
    +---> Sub-task 1 (specialized prompt) → Result 1
    +---> Sub-task 2 (specialized prompt) → Result 2  ← all run in parallel
    +---> Sub-task 3 (specialized prompt) → Result 3
    |
    v
Aggregator → Final Output
```

### Key rules:
- Sub-tasks run **simultaneously** — not sequentially
- Sub-tasks do **not** need to be identical — each can have its own prompt, tools, and criteria
- Aggregator receives all results and makes the final decision

---

## Benefits

| Benefit | Detail |
|---|---|
| **Focused attention** | Claude evaluates one dimension at a time → more thorough analysis |
| **Easier optimisation** | Improve one material's prompt without touching the others |
| **Better scalability** | Add a new material → add one parallel request, nothing else changes |
| **Improved reliability** | Lower cognitive load per request → more consistent results |

---

## When to Use Parallelization

Look for tasks where:
- You're asking Claude to evaluate **multiple independent criteria** simultaneously
- Different evaluations require **different specialised prompts or tools**
- The sub-tasks can **operate independently** of each other
- Speed matters — running in parallel is faster than running sequentially

---

## Key Takeaway

> Parallelization trades one complex prompt for many focused ones running simultaneously.  
> Each sub-task does one thing well. The aggregator synthesises everything.  
> This pattern improves accuracy, maintainability, and scalability.

# Workflow Type: Chaining

---

## What It Is

Break a large, complex task into **smaller sequential subtasks** where each step's
output becomes the next step's input.
```
Input
  |
  v
Processing Task #1
  |
  v
Processing Task #2   ← optional non-LLM processing between steps
  |
  v
Processing Task #3
  |
  v
Output
```

---

## Why Chain Instead of One Big Prompt?

When Claude has too many requirements at once it tends to drop some of them.
Chaining keeps Claude **focused on one aspect at a time**.

- Each task is focused → better results per step
- Non-LLM processing can happen between steps (validation, formatting, API calls)
- Easier to debug — you can see exactly where output breaks down

---

## Real Example — Social Media Video Tool

Instead of one massive prompt, chain the steps:
```
1. Find trending topics on Twitter          ← non-LLM (API call)
2. Select the most interesting topic        ← Claude
3. Research the topic                       ← Claude
4. Write a script for a short video         ← Claude
5. Generate video with AI avatar + TTS      ← non-LLM (AI tools)
6. Post the video to social media           ← non-LLM (API call)
```

Each Claude step is focused on a single task. Non-LLM steps handle the rest.

---

## The Long Prompt Problem — And How Chaining Fixes It

**Problem:** You ask Claude to write a technical article with many constraints:
- Do not mention AI authorship
- No emojis
- No clichéd or casual language
- Professional, technical tone

Even with all constraints stated, Claude may still violate some of them in a
single-pass response.

**Solution — Two-Step Chain:**
```
Step 1: Generate the article
→ Accept that the first draft may not be perfect

Step 2: Send the draft back with focused revision instructions:
"Revise the article below. Follow these steps:
 1. Identify any location that identifies the author as AI and remove it
 2. Find and remove all emojis
 3. Locate any clichéd writing and replace it with technical prose"
```

Step 2 works because Claude can concentrate entirely on **fixing** rather than
trying to both create and constrain simultaneously.

---

## Chaining vs Parallelization

| | Chaining | Parallelization |
|---|---|---|
| **Order** | Sequential — each step depends on the previous | Simultaneous — steps are independent |
| **Use when** | Tasks must happen in order | Tasks can run independently |
| **Example** | Draft → revise → format | Metal eval + Polymer eval + Ceramic eval |

---

## When to Use Chaining

- Complex tasks with many sequential requirements
- Claude consistently ignores some constraints in long single prompts
- You need to validate or process output between steps
- The overall task is too large for one focused prompt

---

## Key Takeaway

> Chaining trades one big unfocused prompt for a series of small focused ones.  
> Each step does one thing well. Steps build on each other sequentially.  
> Use it whenever Claude struggles to satisfy all requirements in a single pass.

# Workflow Type: Routing

---

## What It Is

Use an initial Claude call to **categorize the user's input**, then forward it
to the most appropriate specialised pipeline — not all pipelines, just one.
```
User Input
     |
     v
  Router  ← initial Claude call categorises the request
     |
     +---> Pipeline A (Workflow, Prompt, Tools)
     +---> Pipeline B (Workflow, Prompt, Tools)
     +---> Pipeline C (Workflow, Prompt, Tools)

User input only sent to ONE pipeline
```

---

## The Problem Routing Solves

A generic prompt cannot handle diverse requests equally well.

**Example — video script generator:**

| Topic | Best approach |
|---|---|
| "Python functions" | Educational — clear explanations, relatable examples |
| "Surfing" | Entertainment — high-energy, visual, exciting |
| "Stand-up comedy tips" | Comedy — sharp, unexpected, clever timing |

One prompt trying to handle all three will produce mediocre results for each.

---

## How It Works — Two Steps

### Step 1 — Categorisation
```python
prompt = """
Categorize the topic of a video into one of the listed categories:

<topic>Python functions</topic>

<categories>
- Educational
- Entertainment
- Comedy
- Personal vlog
- Reviews
- Storytelling
</categories>
"""
# Claude responds: "Educational"
```

### Step 2 — Specialised Processing

Use the category to select the right prompt template and generate content:
```python
if category == "Educational":
    prompt = educational_prompt_template
elif category == "Entertainment":
    prompt = entertainment_prompt_template
# ... etc
```

Each template is highly optimised for its category.

---

## Example Category Prompts

| Category | Prompt focus |
|---|---|
| **Educational** | Clear explanations, relatable examples, thought-provoking questions |
| **Entertainment** | High-energy, culturally relevant, trendy language |
| **Comedy** | Sharp observations, unexpected angles, comic timing |
| **Personal vlog** | Authentic, intimate, conversational storytelling |
| **Reviews** | Decisive, experience-based, strengths and weaknesses |
| **Storytelling** | Vivid details, emotional connection, immersive narrative |

---

## When to Use Routing

- Your app handles **diverse request types** that need different approaches
- You can **clearly define categories** that cover your use cases
- Specialised prompts outperform a single generic prompt
- Common in: customer service bots, content generators, support tools

---

## Workflow Patterns — Summary So Far

| Pattern | Structure | Use when |
|---|---|---|
| **Evaluator-Optimizer** | Producer → Grader → loop | Output needs iterative refinement |
| **Parallelization** | Split → run in parallel → aggregate | Independent sub-tasks, speed matters |
| **Chaining** | Sequential steps, each builds on last | Tasks that must happen in order |
| **Routing** | Categorise → forward to one pipeline | Different input types need different handling |

---

## Key Takeaway

> Routing adds one classification step upfront so every subsequent step
> can be highly specialised.  
> The small overhead of categorisation pays off in significantly better
> results for each category of request.

# Agents

---

## What Agents Are

Give Claude a **goal** and a **reasonably abstract set of tools** — Claude figures
out the plan for using those tools to achieve the goal itself.
```
Goal + Tools
      |
      v
   Claude
      |
      v
Plan → Tool calls → Result
```

Unlike workflows where **you** define the steps, in an agent **Claude** defines them.

---

## The Power of Combining Simple Tools

The same small set of tools can handle wildly different tasks:

| Task | Tool calls Claude makes |
|---|---|
| "What's the time?" | `get_current_datetime` |
| "What day is it in 11 days?" | `get_current_datetime` → `add_duration_to_datetime` |
| "Set a reminder for March 21, 2030" | `set_reminder` |
| "What day is my 1000th birthday?" | `add_duration_to_datetime` |
| "Remind me to go to gym next Wednesday" | `get_current_datetime` → `add_duration_to_datetime` → `set_reminder` |
| "When does my 90-day warranty expire?" | Claude asks: "When did you obtain the warranty?" → `add_duration_to_datetime` |

Claude adapts the tool sequence to whatever the task requires — including asking
for missing information when needed.

---

## Tools Should Be Abstract

**Claude Code** demonstrates the right principle:

| Tools Claude Code HAS | Tools Claude Code does NOT have |
|---|---|
| `bash` — run any command | `refactor` — refactor this file |
| `glob` — find files | `run_tests` — discover and run tests |
| `grep` — search file contents | `create_migration` — create a DB migration |
| `ls` — list files | `install_dependencies` — install packages |
| `read` — read a file | |
| `write` — create a file | |
| `edit` — modify a file | |
| `webfetch` — fetch a URL | |

Claude Code **does not** have a "refactor" tool. Instead, it uses `read` +
`edit` + `bash` to accomplish refactoring — a combination the developers never
had to explicitly programme.

> **Rule:** Give Claude general-purpose tools it can combine creatively.
> Hyper-specific tools limit what the agent can do to only what you planned for.

---

## Designing Agent Tools — Example

**Social media video agent toolset:**
```
bash           → access FFMPEG for video processing
generate_image → create images from prompts
text_to_speech → convert text to audio
post_media     → upload to social platforms
```

This set enables both simple workflows (create and post) and dynamic interactions
(generate a sample image → get user approval → proceed with video creation).

---

## Agents vs Workflows — When to Choose

| | Agents | Workflows |
|---|---|---|
| **Steps defined by** | Claude | You (in code) |
| **Flexibility** | High | Low |
| **Reliability** | Harder to guarantee | Easier to guarantee |
| **Cost** | Potentially higher (more tool calls) | More predictable |
| **UX** | More flexible, dynamic | More constrained, predictable |
| **Use when** | Task parameters are unpredictable | You know the exact steps needed |

---

## Key Takeaway

> Agents are powerful because a small set of abstract tools can solve
> a vast range of problems Claude was never explicitly programmed for.  
> The trade-off is reliability and cost — agents are harder to control
> than workflows but far more flexible.  
> Design your tools to be **combinable**, not hyper-specific.

# Environment Inspection

---

## Core Principle

> Claude operates **blindly** — it needs tools to observe the environment
> and understand the results of its own actions.

Without the ability to inspect its environment, Claude cannot:
- Confirm whether an action succeeded
- Understand what the new state looks like
- Detect and correct errors
- Adapt its next step based on what actually happened

---

## Computer Use — The Clearest Example

Every time Claude performs an action (type, click, scroll), the tool call
**automatically returns a screenshot** so Claude can see the new state:
```
Claude calls: computer(action='type', text='Did you read @')
         |
         v
Tool returns: screenshot of current screen
         |
         v
Claude sees the result → decides what to do next
```

Without the screenshot, Claude would have no idea if the text appeared,
if a dropdown opened, or if the page changed entirely.

---

## File Operations — Read Before Writing

The same principle applies to files. Before modifying any file, Claude must
read it first to understand the current structure:
```
Task: "Add a new route to this Python file"
         |
         v
Claude calls: read(file)         ← inspect first
         |
         v
Claude sees current code structure
         |
         v
Claude calls: edit(file, changes) ← now safe to modify
```

Modifying without reading risks breaking existing functionality.

---

## Designing for Environment Inspection

When building any agent, always ask:
**"How will Claude know if this action worked?"**

| Context | How to enable inspection |
|---|---|
| File operations | Read file before and after modification |
| UI interactions | Return screenshot after each action |
| API calls | Check and return the full response |
| Video generation | Extract screenshots with FFmpeg; run Whisper to verify audio |
| Content creation | Validate output against original requirements before finishing |

---

## System Prompt Guidance Example — Video Agent
```
When generating video content:
1. Use bash to run whisper.cpp and generate caption files with
   timestamps to verify dialogue placement
2. Use FFmpeg to extract screenshots at regular intervals to
   visually inspect the output
3. Compare generated content against the original requirements
   before marking the task complete
```

You can instruct Claude to inspect its own output through the system prompt.

---

## Benefits

| Benefit | Detail |
|---|---|
| **Progress tracking** | Claude knows how close it is to finishing |
| **Error detection** | Unexpected results are caught before cascading |
| **Quality assurance** | Output verified before task is marked complete |
| **Adaptive behaviour** | Claude adjusts approach based on observed state |

---

## Key Takeaway

> Environment inspection transforms Claude from a blind command executor
> into an agent that genuinely understands its working environment.  
> Always design agents with observation loops —
> every significant action should be followed by a way to see its result.|

# Workflows vs Agents — When to Use Each

---

## Quick Definition Recap

| | Workflows | Agents |
|---|---|---|
| **What it is** | Predefined series of Claude calls through known steps | Claude gets a goal + tools and formulates its own plan |
| **Who defines the steps** | You (in code) | Claude (at runtime) |
| **Use when** | You can picture the exact flow ahead of time | You don't know what tasks or parameters Claude will face |

---

## Benefits

| Workflows | Agents |
|---|---|
| Claude focuses on one subtask at a time → higher accuracy | Flexible UX — handles open-ended requests |
| Easy to evaluate and test — you know each step | Can combine tools in unexpected ways |
| Predictable and reliable execution | Handles novel situations not anticipated during development |
| Well-suited for specific, well-defined problems | Can ask users for missing information when needed |

---

## Downsides

| Workflows | Agents |
|---|---|
| Less flexible — built for specific task types | Lower task completion rate than workflows |
| Constrained UX — requires known inputs | Harder to test and evaluate — steps are unpredictable |
| More upfront design and planning required | Less predictable behaviour in production |

---

## The Engineer's Guiding Rule

> Users don't care that you built a fancy agent.  
> They want a product that **works consistently**.
```
Always implement workflows where possible.
Only use agents when they are truly required.
```

---

## Decision Guide
```
Can you picture the exact steps needed to complete the task?
        |
       Yes → Workflow
        |
        No
        |
        v
Are the task types and parameters unpredictable?
        |
       Yes → Agent
        |
        No → Reconsider — you may be able to define a workflow
```

---

## Practical Examples

| Scenario | Use |
|---|---|
| Image → CAD file with known steps | Workflow |
| Convert document → markdown with review step | Workflow |
| Categorise input → route to specialised pipeline | Workflow (routing) |
| Open-ended coding assistant (Claude Code) | Agent |
| General-purpose chatbot with tools | Agent |
| Customer service bot with fixed issue types | Workflow |
| Customer service bot with unlimited query types | Agent |

---

## Key Takeaway

> Workflows give you **reliability and predictability**.  
> Agents give you **flexibility and adaptability**.  
> Start with workflows. Add agent flexibility only when the problem
> genuinely requires it.